## **Ridge Regularization From Scratch (OLS Method)**

### **Topic Roadmap**

 **1. Imports & Setup**

 **2. Dataset Preparation**

 **3. Scikit-Learn Ridge Baseline**

 **4. Custom Ridge Implementation (Closed-Form)**

 **5. Model Evaluation**

### **1. Imports & Setup**

Import the required libraries for data handling, modeling, and evaluation.

In [1]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

### **2. Dataset Preparation**

Load the diabetes dataset and split it into training and testing sets.

The training set trains the model, while the testing set provides an objective measure of generalization.

In [2]:
# Load dataset
X, y = load_diabetes(return_X_y=True)

# Create training and testing splits
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4)

### **3. Scikit-Learn Ridge Baseline**

Train a standard Scikit-Learn Ridge Regression model to establish a performance baseline.

The `cholesky` solver computes the standard closed-form solution.

In [3]:
# Initialize and train the baseline model
ridge_baseline = Ridge(alpha=0.1, solver='cholesky')
ridge_baseline.fit(X_train, y_train)

# Evaluate baseline performance
y_pred_baseline = ridge_baseline.predict(X_test)
baseline_r2 = r2_score(y_test, y_pred_baseline)

print(f"Baseline Ridge R2 Score: {baseline_r2:.4f}")

Baseline Ridge R2 Score: 0.4693


### **4. Custom Ridge Implementation (Closed-Form)**

Build a custom Ridge regressor using the Ordinary Least Squares (OLS) closed-form formula.

This uses the Normal Equation with an $L2$ penalty: $W = (X^T X + \alpha I)^{-1} X^T Y$. The identity matrix $I$ is modified to leave the intercept unpenalized.

In [4]:
class CustomRidge:
    def __init__(self, alpha=0.1):
        self.alpha = alpha
        self.coef_ = None
        self.intercept_ = None
        
    def fit(self, X_train, y_train):
        # Add a column of 1s to account for the intercept term
        X_train_bias = np.insert(X_train, 0, 1, axis=1)
        
        # Create an Identity matrix for the L2 penalty
        I = np.identity(X_train_bias.shape[1])
        
        # Do not regularize the intercept (first element)
        I[0][0] = 0
        
        # Apply the closed-form Ridge Normal Equation: W = (X^T * X + alpha * I)^-1 * X^T * Y
        XT_X = np.dot(X_train_bias.T, X_train_bias)
        penalty = self.alpha * I
        XT_y = np.dot(X_train_bias.T, y_train)
        
        # Calculate weights
        result = np.linalg.inv(XT_X + penalty).dot(XT_y)
        
        # Extract intercept and coefficients
        self.intercept_ = result[0]
        self.coef_ = result[1:]
    
    def predict(self, X_test):
        return np.dot(X_test, self.coef_) + self.intercept_

### **5. Model Evaluation**

Train the custom Ridge model and evaluate its performance to ensure mathematical equivalence with the Scikit-Learn implementation.

In [5]:
# Initialize and train the custom model
custom_ridge = CustomRidge(alpha=0.1)
custom_ridge.fit(X_train, y_train)

# Evaluate custom model performance
y_pred_custom = custom_ridge.predict(X_test)
custom_r2 = r2_score(y_test, y_pred_custom)

print(f"Custom Ridge R2 Score: {custom_r2:.4f}")

Custom Ridge R2 Score: 0.4693


## **Key Revision Notes**

- **Closed-Form Ridge Formula:** The mathematical solution for Ridge Regression via Ordinary Least Squares is defined as $W = (X^T X + \alpha I)^{-1} X^T Y$.
- **Identity Matrix Modification:** Regularization penalizes feature weights but should **never penalize the intercept**. To handle this computationally, the top-left element of the Identity matrix ($I_{0,0}$) is explicitly set to `0`.
- **Feature Augmentation:** To solve for the intercept alongside the coefficients in a single matrix operation, an array of ones is prefixed to the feature matrix $X$.